# Gridlock - Traffic Demand Prediction (Optimized)
## Target: 94-95 R² Score

In [ ]:
!pip install lightgbm xgboost catboost geohash2 optuna scikit-learn pandas numpy -q

In [ ]:
import pandas as pd
import numpy as np
import warnings, os, re
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.metrics import r2_score
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               ExtraTreesRegressor, StackingRegressor,
                               VotingRegressor, BaggingRegressor)
from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.neighbors import KNeighborsRegressor

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import geohash2

import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use("ggplot")

SEED = 42
np.random.seed(SEED)
print("All imports successful!")

In [ ]:
TRAIN_PATH = "dataset/train.csv"
TEST_PATH  = "dataset/test.csv"
SUB_PATH   = "dataset/sample_submission.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sub = pd.read_csv(SUB_PATH)
print("Train shape:", train.shape)
print("Test  shape:", test.shape)
print("\nTrain columns:", train.columns.tolist())
print("\nMissing values (train):")
print(train.isnull().sum())
print("\nTarget stats:")
print(train["demand"].describe())
train.head()

In [ ]:
def decode_geohash(gh):
    try:
        lat, lon, _, _ = geohash2.decode_exactly(str(gh))
        return float(lat), float(lon)
    except Exception:
        return np.nan, np.nan

In [ ]:
def parse_timestamp(ts_series):
    hours, minutes = [], []
    for v in ts_series:
        v = str(v).strip()
        if ":" in v:
            parts = v.split(":")
            h = int(parts[0]) if parts[0].isdigit() else 0
            m = int(parts[1]) if len(parts) > 1 and parts[1].strip().isdigit() else 0
        else:
            try:
                frac = float(v)
                h = int(frac * 24)
                m = int((frac * 24 - h) * 60)
            except Exception:
                h, m = 0, 0
        hours.append(h)
        minutes.append(m)
    return pd.Series(hours, index=ts_series.index), pd.Series(minutes, index=ts_series.index)

def cyclical_encode(series, max_val):
    sin_vals = np.sin(2 * np.pi * series / max_val)
    cos_vals = np.cos(2 * np.pi * series / max_val)
    return sin_vals, cos_vals

In [ ]:
def feature_engineering(df, is_train=True,
                         geo_stats=None, geo3_stats=None, geo5_stats=None,
                         target_enc=None, target_enc_hour=None,
                         target_enc_road=None, target_enc_weather=None,
                         road_mode="Residential", weather_mode="Sunny",
                         temp_median=None,
                         geo_hour_stats=None, geo_day_stats=None,
                         geo_road_stats=None, geo_weather_stats=None,
                         hour_stats=None):
    df = df.copy()

    # ── 1. Decode geohash → lat/lon
    decoded = df["geohash"].apply(decode_geohash)
    df["lat"] = decoded.apply(lambda x: x[0])
    df["lon"] = decoded.apply(lambda x: x[1])
    df["geo_prefix3"] = df["geohash"].str[:3]
    df["geo_prefix4"] = df["geohash"].str[:4]
    df["geo_prefix5"] = df["geohash"].str[:5]

    # Geohash length as feature
    df["geohash_len"] = df["geohash"].str.len()

    # ── 2. Timestamp features
    df["hour"], df["minute"] = parse_timestamp(df["timestamp"])
    df["time_slot"] = df["hour"] * 4 + df["minute"] // 15  # 96 slots per day
    df["hour_sin"], df["hour_cos"] = cyclical_encode(df["hour"], 24)
    df["minute_sin"], df["minute_cos"] = cyclical_encode(df["minute"], 60)
    df["time_slot_sin"], df["time_slot_cos"] = cyclical_encode(df["time_slot"], 96)

    # Time-of-day buckets (more granular)
    df["is_night"]       = ((df["hour"] >= 22) | (df["hour"] < 6)).astype(int)
    df["is_morning"]     = ((df["hour"] >= 6) & (df["hour"] < 10)).astype(int)
    df["is_midday"]      = ((df["hour"] >= 10) & (df["hour"] < 14)).astype(int)
    df["is_afternoon"]   = ((df["hour"] >= 14) & (df["hour"] < 18)).astype(int)
    df["is_evening"]     = ((df["hour"] >= 18) & (df["hour"] < 22)).astype(int)
    df["is_rush_hour"]   = (((df["hour"] >= 7) & (df["hour"] <= 9)) |
                            ((df["hour"] >= 16) & (df["hour"] <= 19))).astype(int)
    df["is_early_morning"] = ((df["hour"] >= 4) & (df["hour"] < 7)).astype(int)
    df["is_late_night"]  = ((df["hour"] >= 0) & (df["hour"] < 4)).astype(int)

    # Day features
    df["is_day49"] = (df["day"] == 49).astype(int)

    # ── 3. Temperature features
    temp_median_val = temp_median if temp_median is not None else df["Temperature"].median()
    if is_train:
        temp_median = temp_median_val
    df["temp_missing"]   = df["Temperature"].isna().astype(int)
    df["Temperature"]    = df["Temperature"].fillna(temp_median_val)
    df["temp_sq"]        = df["Temperature"] ** 2
    df["temp_abs"]       = df["Temperature"].abs()
    df["is_cold"]        = (df["Temperature"] < 5).astype(int)
    df["is_mild"]        = ((df["Temperature"] >= 5) & (df["Temperature"] <= 20)).astype(int)
    df["is_warm"]        = ((df["Temperature"] > 20) & (df["Temperature"] <= 30)).astype(int)
    df["is_hot"]         = (df["Temperature"] > 30).astype(int)
    df["temp_bin"]       = pd.cut(df["Temperature"], bins=10, labels=False).fillna(0).astype(int)
    df["is_freezing"]    = (df["Temperature"] < 0).astype(int)

    # ── 4. RoadType
    df["road_missing"] = df["RoadType"].isna().astype(int)
    df["RoadType"] = df["RoadType"].fillna(road_mode)
    road_dummies = pd.get_dummies(df["RoadType"], prefix="road").astype(int)
    for col in ["road_Residential", "road_Street", "road_Highway"]:
        df[col] = road_dummies.get(col, 0)

    # ── 5. Weather
    df["weather_missing"] = df["Weather"].isna().astype(int)
    df["Weather"] = df["Weather"].fillna(weather_mode)
    weather_map = {"Sunny": 0, "Rainy": 2, "Foggy": 1, "Snowy": 3}
    df["Weather_enc"] = df["Weather"].map(weather_map).fillna(0).astype(int)

    for w in ["Sunny", "Rainy", "Foggy", "Snowy"]:
        df[f"weather_{w}"] = (df["Weather"] == w).astype(int)

    # ── 6. Binary features
    df["LargeVehicles_bin"] = (df["LargeVehicles"].str.strip() == "Allowed").astype(int)
    df["Landmarks_bin"] = (df["Landmarks"].str.strip() == "Yes").astype(int)
    df["NumberofLanes"] = pd.to_numeric(df["NumberofLanes"], errors="coerce").fillna(1).astype(int)
    df["lanes_sq"] = df["NumberofLanes"] ** 2
    df["lanes_log"] = np.log1p(df["NumberofLanes"])

    # ── 7. INTERACTION FEATURES (many more)
    df["lanes_x_weather"]    = df["NumberofLanes"] * df["Weather_enc"]
    df["temp_x_weather"]     = df["Temperature"] * df["Weather_enc"]
    df["hour_x_day"]         = df["hour"] * df["day"]
    df["geo_x_hour"]         = df["lat"] * df["hour_sin"]
    df["landmark_x_lane"]    = df["Landmarks_bin"] * df["NumberofLanes"]
    df["rain_x_temp"]        = df["weather_Rainy"] * df["Temperature"]
    df["highway_x_lane"]     = df["road_Highway"] * df["NumberofLanes"]
    df["night_x_lane"]       = df["is_night"] * df["NumberofLanes"]

    # NEW powerful interactions
    df["lat_x_lon"]          = df["lat"] * df["lon"]
    df["lat_sq"]             = df["lat"] ** 2
    df["lon_sq"]             = df["lon"] ** 2
    df["rush_x_lane"]        = df["is_rush_hour"] * df["NumberofLanes"]
    df["rush_x_highway"]     = df["is_rush_hour"] * df["road_Highway"]
    df["rush_x_temp"]        = df["is_rush_hour"] * df["Temperature"]
    df["large_x_highway"]    = df["LargeVehicles_bin"] * df["road_Highway"]
    df["large_x_lane"]       = df["LargeVehicles_bin"] * df["NumberofLanes"]
    df["landmark_x_highway"] = df["Landmarks_bin"] * df["road_Highway"]
    df["snow_x_temp"]        = df["weather_Snowy"] * df["Temperature"]
    df["fog_x_hour"]         = df["weather_Foggy"] * df["hour"]
    df["rain_x_rush"]        = df["weather_Rainy"] * df["is_rush_hour"]
    df["morning_x_lane"]     = df["is_morning"] * df["NumberofLanes"]
    df["evening_x_lane"]     = df["is_evening"] * df["NumberofLanes"]
    df["time_slot_x_lane"]   = df["time_slot"] * df["NumberofLanes"]
    df["lat_x_hour"]         = df["lat"] * df["hour"]
    df["lon_x_hour"]         = df["lon"] * df["hour"]
    df["temp_x_hour"]        = df["Temperature"] * df["hour"]
    df["temp_x_lane"]        = df["Temperature"] * df["NumberofLanes"]
    df["day_x_slot"]         = df["day"] * df["time_slot"]
    df["lane_x_weather_x_hour"] = df["NumberofLanes"] * df["Weather_enc"] * df["hour"]
    df["landmark_x_rush"]    = df["Landmarks_bin"] * df["is_rush_hour"]

    # Distance from centroid
    df["dist_from_center"]   = np.sqrt((df["lat"] - df["lat"].mean())**2 +
                                       (df["lon"] - df["lon"].mean())**2)

    # ── 8. GEO AGGREGATION FEATURES
    if is_train:
        geo_stats = (
            df.groupby("geo_prefix4")["demand"]
              .agg(geo4_mean="mean", geo4_std="std",
                   geo4_med="median", geo4_cnt="count",
                   geo4_min="min", geo4_max="max")
              .reset_index()
        )
        geo_stats["geo4_std"] = geo_stats["geo4_std"].fillna(0)
        geo_stats["geo4_range"] = geo_stats["geo4_max"] - geo_stats["geo4_min"]

        geo3_stats = (
            df.groupby("geo_prefix3")["demand"]
              .agg(geo3_mean="mean", geo3_std="std", geo3_cnt="count")
              .reset_index()
        )

        geo5_stats = (
            df.groupby("geo_prefix5")["demand"]
              .agg(geo5_mean="mean", geo5_std="std", geo5_cnt="count")
              .reset_index()
        )

        # Geohash x Hour interaction stats
        geo_hour_stats = (
            df.groupby(["geo_prefix4", "hour"])["demand"]
              .agg(geo_hour_mean="mean", geo_hour_std="std")
              .reset_index()
        )
        geo_hour_stats["geo_hour_std"] = geo_hour_stats["geo_hour_std"].fillna(0)

        # Geohash x Day stats
        geo_day_stats = (
            df.groupby(["geo_prefix4", "day"])["demand"]
              .agg(geo_day_mean="mean", geo_day_std="std")
              .reset_index()
        )

        # Hour-level stats (demand pattern across hours)
        hour_stats = (
            df.groupby("hour")["demand"]
              .agg(hour_mean="mean", hour_std="std", hour_med="median")
              .reset_index()
        )

        # Geo x RoadType stats
        geo_road_stats = (
            df.groupby(["geo_prefix4", "RoadType"])["demand"]
              .agg(geo_road_mean="mean")
              .reset_index()
        )

        # Geo x Weather stats
        geo_weather_stats = (
            df.groupby(["geo_prefix4", "Weather"])["demand"]
              .agg(geo_weather_mean="mean")
              .reset_index()
        )

    # Merge all geo stats
    df = df.merge(geo_stats[["geo_prefix4", "geo4_mean", "geo4_std", "geo4_med",
                              "geo4_cnt", "geo4_min", "geo4_max", "geo4_range"]],
                  on="geo_prefix4", how="left")
    df = df.merge(geo3_stats, on="geo_prefix3", how="left")
    df = df.merge(geo5_stats, on="geo_prefix5", how="left")
    df = df.merge(geo_hour_stats, on=["geo_prefix4", "hour"], how="left")
    df = df.merge(geo_day_stats, on=["geo_prefix4", "day"], how="left")
    df = df.merge(hour_stats, on="hour", how="left")
    df = df.merge(geo_road_stats, on=["geo_prefix4", "RoadType"], how="left")
    df = df.merge(geo_weather_stats, on=["geo_prefix4", "Weather"], how="left")

    # Fill NaNs in aggregations
    for col in ["geo4_mean", "geo4_std", "geo4_med", "geo4_cnt", "geo4_min",
                "geo4_max", "geo4_range",
                "geo3_mean", "geo3_std", "geo3_cnt",
                "geo5_mean", "geo5_std", "geo5_cnt",
                "geo_hour_mean", "geo_hour_std",
                "geo_day_mean", "geo_day_std",
                "hour_mean", "hour_std", "hour_med",
                "geo_road_mean", "geo_weather_mean"]:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # Deviation features
    df["demand_dev_from_geo4"]  = 0  # placeholder for test
    df["demand_dev_from_hour"]  = 0

    # ── 9. TARGET ENCODING (per-geohash mean demand) - leak-free for train
    if is_train:
        target_enc = (df.groupby("geohash")["demand"]
                         .mean()
                         .rename("geo_target_mean")
                         .reset_index())

        target_enc_hour = (df.groupby(["geohash", "hour"])["demand"]
                              .mean()
                              .rename("geo_hour_target")
                              .reset_index())

        target_enc_road = (df.groupby("RoadType")["demand"]
                              .mean()
                              .rename("road_target_mean")
                              .reset_index())

        target_enc_weather = (df.groupby("Weather")["demand"]
                                 .mean()
                                 .rename("weather_target_mean")
                                 .reset_index())

    df = df.merge(target_enc, on="geohash", how="left")
    df["geo_target_mean"] = df["geo_target_mean"].fillna(
        df["geo_target_mean"].mean() if not df["geo_target_mean"].isna().all() else 0
    )

    df = df.merge(target_enc_hour, on=["geohash", "hour"], how="left")
    df["geo_hour_target"] = df["geo_hour_target"].fillna(df["geo_target_mean"])

    df = df.merge(target_enc_road, on="RoadType", how="left")
    df["road_target_mean"] = df["road_target_mean"].fillna(0)

    df = df.merge(target_enc_weather, on="Weather", how="left")
    df["weather_target_mean"] = df["weather_target_mean"].fillna(0)

    # Ratio features from target encoding
    df["geo_vs_hour_ratio"] = df["geo_target_mean"] / (df["hour_mean"] + 1e-8)
    df["geo_vs_global_ratio"] = df["geo_target_mean"] / (df["geo4_mean"] + 1e-8)

    # ── 10. Frequency encoding
    geo_freq = df["geohash"].value_counts().to_dict()
    df["geo_freq"] = df["geohash"].map(geo_freq)

    geo4_freq = df["geo_prefix4"].value_counts().to_dict()
    df["geo4_freq"] = df["geo_prefix4"].map(geo4_freq)

    return (df, geo_stats, geo3_stats, geo5_stats,
            target_enc, target_enc_hour, target_enc_road, target_enc_weather,
            temp_median,
            geo_hour_stats, geo_day_stats, geo_road_stats, geo_weather_stats,
            hour_stats)

print("Feature engineering function defined!")

In [ ]:
(train_fe, geo_stats, geo3_stats, geo5_stats,
 target_enc, target_enc_hour, target_enc_road, target_enc_weather,
 temp_med,
 geo_hour_stats, geo_day_stats, geo_road_stats, geo_weather_stats,
 hour_stats) = feature_engineering(train, is_train=True)

(test_fe, *_) = feature_engineering(
    test, is_train=False,
    geo_stats=geo_stats, geo3_stats=geo3_stats, geo5_stats=geo5_stats,
    target_enc=target_enc, target_enc_hour=target_enc_hour,
    target_enc_road=target_enc_road, target_enc_weather=target_enc_weather,
    temp_median=temp_med,
    geo_hour_stats=geo_hour_stats, geo_day_stats=geo_day_stats,
    geo_road_stats=geo_road_stats, geo_weather_stats=geo_weather_stats,
    hour_stats=hour_stats)

print("Train FE shape:", train_fe.shape)
print("Test  FE shape:", test_fe.shape)

In [ ]:
DROP_COLS = ["Index", "geohash", "timestamp", "RoadType", "Weather",
             "LargeVehicles", "Landmarks",
             "geo_prefix3", "geo_prefix4", "geo_prefix5",
             "demand", "demand_dev_from_geo4", "demand_dev_from_hour"]
FEATURE_COLS = [c for c in train_fe.columns
                if c not in DROP_COLS and c in test_fe.columns]

X      = train_fe[FEATURE_COLS].fillna(0)
X_test = test_fe[FEATURE_COLS].reindex(columns=FEATURE_COLS, fill_value=0)

y_raw = train_fe["demand"].copy()
y     = np.log1p(y_raw)

print(f"Feature count : {X.shape[1]}")
print(f"y range       : {y_raw.min():.6f} -> {y_raw.max():.6f}")
print(f"log1p(y) range: {y.min():.4f} -> {y.max():.4f}")
print(f"\nFeature names ({len(FEATURE_COLS)}):")
for i, c in enumerate(FEATURE_COLS):
    print(f"  {i+1:3d}. {c}")

In [ ]:
def competition_score(y_true_raw, y_pred_log):
    y_pred_raw = np.expm1(y_pred_log)
    y_pred_raw = np.clip(y_pred_raw, 0, None)
    return max(0, 100 * r2_score(y_true_raw, y_pred_raw))

def cv_oof(model, X, y, y_raw, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X))
    scores = []
    for tr_idx, val_idx in kf.split(X):
        m = type(model)(**model.get_params())
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        oof[val_idx] = m.predict(X.iloc[val_idx])
        scores.append(competition_score(y_raw.iloc[val_idx], oof[val_idx]))
    return oof, np.mean(scores), np.std(scores)

print("Scoring functions defined!")

In [ ]:
print("=" * 60)
print("BASELINE MODEL COMPARISON  (5-fold CV, log1p target)")
print("=" * 60)

baseline_models = {
    "RandomForest" : RandomForestRegressor(n_estimators=500, max_depth=20,
                                            min_samples_leaf=5, random_state=SEED, n_jobs=-1),
    "ExtraTrees"   : ExtraTreesRegressor(n_estimators=500, max_depth=20,
                                           min_samples_leaf=5, random_state=SEED, n_jobs=-1),
    "LightGBM"     : lgb.LGBMRegressor(n_estimators=800, learning_rate=0.05,
                                         num_leaves=127, random_state=SEED,
                                         n_jobs=-1, verbose=-1),
    "XGBoost"      : xgb.XGBRegressor(n_estimators=800, learning_rate=0.05,
                                        max_depth=7, random_state=SEED,
                                        tree_method="hist", n_jobs=-1, verbosity=0),
    "CatBoost"     : cb.CatBoostRegressor(iterations=800, learning_rate=0.05,
                                           depth=7, random_seed=SEED, verbose=0),
}

baseline_scores = {}
for name, model in baseline_models.items():
    _, mean_s, std_s = cv_oof(model, X, y, y_raw, n_splits=5)
    baseline_scores[name] = mean_s
    print(f"{name:15s}  Score: {mean_s:.4f} +/- {std_s:.4f}")

In [ ]:
def lgb_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int("n_estimators",     500, 3000),
        learning_rate     = trial.suggest_float("learning_rate",    0.005, 0.1, log=True),
        num_leaves        = trial.suggest_int("num_leaves",       31, 512),
        max_depth         = trial.suggest_int("max_depth",        4, 15),
        min_child_samples = trial.suggest_int("min_child_samples", 3, 100),
        subsample         = trial.suggest_float("subsample",        0.5, 1.0),
        colsample_bytree  = trial.suggest_float("colsample_bytree", 0.4, 1.0),
        reg_alpha         = trial.suggest_float("reg_alpha",        1e-5, 10.0, log=True),
        reg_lambda        = trial.suggest_float("reg_lambda",       1e-5, 10.0, log=True),
        min_split_gain    = trial.suggest_float("min_split_gain",   0.0, 1.0),
        random_state=SEED, n_jobs=-1, verbose=-1,
    )
    m = lgb.LGBMRegressor(**params)
    _, s, _ = cv_oof(m, X, y, y_raw, n_splits=5)
    return s

print("\nTuning LightGBM (100 trials)...")
lgb_study = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
lgb_study.optimize(lgb_objective, n_trials=100, show_progress_bar=True)
best_lgb = {**lgb_study.best_params,
            "random_state": SEED, "n_jobs": -1, "verbose": -1}
print(f"Best LGB score : {lgb_study.best_value:.4f}")
print(f"Best LGB params: {best_lgb}")

In [ ]:
def xgb_objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int("n_estimators",     500, 3000),
        learning_rate    = trial.suggest_float("learning_rate",    0.005, 0.1, log=True),
        max_depth        = trial.suggest_int("max_depth",        3, 12),
        min_child_weight = trial.suggest_int("min_child_weight", 1, 30),
        subsample        = trial.suggest_float("subsample",        0.5, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.4, 1.0),
        reg_alpha        = trial.suggest_float("reg_alpha",        1e-5, 10.0, log=True),
        reg_lambda       = trial.suggest_float("reg_lambda",       1e-5, 10.0, log=True),
        gamma            = trial.suggest_float("gamma",            0, 5),
        random_state=SEED, tree_method="hist", n_jobs=-1, verbosity=0,
    )
    m = xgb.XGBRegressor(**params)
    _, s, _ = cv_oof(m, X, y, y_raw, n_splits=5)
    return s

print("\nTuning XGBoost (100 trials)...")
xgb_study = optuna.create_study(direction="maximize",
                                  sampler=optuna.samplers.TPESampler(seed=SEED))
xgb_study.optimize(xgb_objective, n_trials=100, show_progress_bar=True)
best_xgb = {**xgb_study.best_params,
            "random_state": SEED, "tree_method": "hist", "n_jobs": -1, "verbosity": 0}
print(f"Best XGB score : {xgb_study.best_value:.4f}")

In [ ]:
def cat_objective(trial):
    params = dict(
        iterations        = trial.suggest_int("iterations",        500, 3000),
        learning_rate     = trial.suggest_float("learning_rate",     0.005, 0.1, log=True),
        depth             = trial.suggest_int("depth",             4, 10),
        l2_leaf_reg       = trial.suggest_float("l2_leaf_reg",       1e-3, 10.0, log=True),
        bagging_temperature = trial.suggest_float("bagging_temperature", 0, 1),
        border_count      = trial.suggest_int("border_count",      32, 255),
        random_seed=SEED, verbose=0,
    )
    m = cb.CatBoostRegressor(**params)
    _, s, _ = cv_oof(m, X, y, y_raw, n_splits=5)
    return s

print("\nTuning CatBoost (80 trials)...")
cat_study = optuna.create_study(direction="maximize",
                                  sampler=optuna.samplers.TPESampler(seed=SEED))
cat_study.optimize(cat_objective, n_trials=80, show_progress_bar=True)
best_cat = {**cat_study.best_params,
            "random_seed": SEED, "verbose": 0}
print(f"Best CAT score : {cat_study.best_value:.4f}")

In [ ]:
def et_objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int("n_estimators",     300, 2000),
        max_depth        = trial.suggest_int("max_depth",        10, 40),
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20),
        min_samples_leaf  = trial.suggest_int("min_samples_leaf",  1, 20),
        max_features     = trial.suggest_float("max_features",    0.3, 1.0),
        random_state=SEED, n_jobs=-1,
    )
    m = ExtraTreesRegressor(**params)
    _, s, _ = cv_oof(m, X, y, y_raw, n_splits=5)
    return s

print("\nTuning ExtraTrees (50 trials)...")
et_study = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
et_study.optimize(et_objective, n_trials=50, show_progress_bar=True)
best_et = {**et_study.best_params,
           "random_state": SEED, "n_jobs": -1}
print(f"Best ET score : {et_study.best_value:.4f}")

In [ ]:
tuned_models = {
    "LGB_tuned" : lgb.LGBMRegressor(**best_lgb),
    "XGB_tuned" : xgb.XGBRegressor(**best_xgb),
    "CAT_tuned" : cb.CatBoostRegressor(**best_cat),
    "ET_tuned"  : ExtraTreesRegressor(**best_et),
}

# Multi-seed ensemble for more robust predictions
SEEDS = [42, 123, 456, 789, 2024]
N_SPLITS = 7  # more folds = more stable OOF

all_oof_preds = {n: np.zeros(len(X)) for n in tuned_models}
all_test_preds = {n: np.zeros(len(X_test)) for n in tuned_models}

print("\nBuilding multi-seed OOF predictions...")
for seed_idx, seed in enumerate(SEEDS):
    print(f"\n  Seed {seed} ({seed_idx+1}/{len(SEEDS)}):")
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
        Xtr, Xval = X.iloc[tr_idx], X.iloc[val_idx]
        ytr, yval = y.iloc[tr_idx], y.iloc[val_idx]
        
        for name, model in tuned_models.items():
            params = model.get_params()
            # Update random state for diversity
            if "random_state" in params:
                params["random_state"] = seed
            elif "random_seed" in params:
                params["random_seed"] = seed
            m = type(model)(**params)
            m.fit(Xtr, ytr)
            all_oof_preds[name][val_idx] += m.predict(Xval) / len(SEEDS)
            all_test_preds[name] += m.predict(X_test) / (N_SPLITS * len(SEEDS))
        
    print(f"    Done")

print("\n=== Multi-seed OOF Scores ===")
oof_scores = {}
for name in tuned_models:
    s = competition_score(y_raw, all_oof_preds[name])
    oof_scores[name] = s
    print(f"  {name:15s}  {s:.4f}")

In [ ]:
# ── Weighted blend (weights proportional to OOF score)
w_arr = np.array([oof_scores[n] for n in tuned_models])
weights = w_arr / w_arr.sum()
print("\nBlend weights:")
for n, w in zip(tuned_models, weights):
    print(f"  {n:15s}: {w:.4f}")

blend_oof  = sum(w * all_oof_preds[n] for w, n in zip(weights, tuned_models))
blend_test = sum(w * all_test_preds[n] for w, n in zip(weights, tuned_models))
print(f"\nWeighted blend OOF: {competition_score(y_raw, blend_oof):.4f}")

# ── Optimized blend (search for best weights)
from scipy.optimize import minimize

def neg_score(w):
    w = np.array(w)
    w = w / w.sum()
    pred = sum(wi * all_oof_preds[n] for wi, n in zip(w, tuned_models))
    return -competition_score(y_raw, pred)

result = minimize(neg_score,
                  x0=np.ones(len(tuned_models)) / len(tuned_models),
                  method="Nelder-Mead",
                  options={"maxiter": 10000})
opt_weights = np.array(result.x)
opt_weights = opt_weights / opt_weights.sum()

print("\nOptimized weights:")
for n, w in zip(tuned_models, opt_weights):
    print(f"  {n:15s}: {w:.4f}")

opt_blend_oof  = sum(w * all_oof_preds[n] for w, n in zip(opt_weights, tuned_models))
opt_blend_test = sum(w * all_test_preds[n] for w, n in zip(opt_weights, tuned_models))
print(f"Optimized blend OOF: {competition_score(y_raw, opt_blend_oof):.4f}")

# ── Ridge meta-learner (stacking)
oof_stack = np.column_stack([all_oof_preds[n] for n in tuned_models])
test_stack = np.column_stack([all_test_preds[n] for n in tuned_models])

# CV-based stacking to avoid overfitting
from sklearn.model_selection import cross_val_predict

meta = Ridge(alpha=1.0)
stacked_oof = cross_val_predict(meta, oof_stack, y, cv=KFold(5, shuffle=True, random_state=SEED))
meta.fit(oof_stack, y)
stacked_test = meta.predict(test_stack)
print(f"Ridge-stacked OOF: {competition_score(y_raw, stacked_oof):.4f}")

# ── BayesianRidge meta-learner
meta_br = BayesianRidge()
stacked_br_oof = cross_val_predict(meta_br, oof_stack, y, cv=KFold(5, shuffle=True, random_state=SEED))
meta_br.fit(oof_stack, y)
stacked_br_test = meta_br.predict(test_stack)
print(f"BayesianRidge-stacked OOF: {competition_score(y_raw, stacked_br_oof):.4f}")

# ── ElasticNet meta-learner
meta_en = ElasticNet(alpha=0.001, l1_ratio=0.5)
stacked_en_oof = cross_val_predict(meta_en, oof_stack, y, cv=KFold(5, shuffle=True, random_state=SEED))
meta_en.fit(oof_stack, y)
stacked_en_test = meta_en.predict(test_stack)
print(f"ElasticNet-stacked OOF: {competition_score(y_raw, stacked_en_oof):.4f}")

In [ ]:
candidates = {
    **{n: (all_oof_preds[n], all_test_preds[n]) for n in tuned_models},
    "blend"         : (blend_oof, blend_test),
    "opt_blend"     : (opt_blend_oof, opt_blend_test),
    "ridge_stacked" : (stacked_oof, stacked_test),
    "bayes_stacked" : (stacked_br_oof, stacked_br_test),
    "enet_stacked"  : (stacked_en_oof, stacked_en_test),
}

print("\n=== ALL CANDIDATE SCORES ===")
best_name, best_score = None, -np.inf
for cname, (oof_p, _) in candidates.items():
    s = competition_score(y_raw, oof_p)
    marker = " <<< BEST" if s > best_score else ""
    print(f"  {cname:20s}  {s:.4f}{marker}")
    if s > best_score:
        best_score, best_name = s, cname

print(f"\n>>> Chosen: {best_name}  (score={best_score:.4f})")

# Final predictions
final_test_pred = np.expm1(candidates[best_name][1])
final_test_pred = np.clip(final_test_pred, 0, None)

In [ ]:
# Post-processing: blend best candidate with runner-up
all_scores = {}
for cname, (oof_p, _) in candidates.items():
    all_scores[cname] = competition_score(y_raw, oof_p)

sorted_candidates = sorted(all_scores.items(), key=lambda x: -x[1])
print("Ranking:")
for rank, (name, score) in enumerate(sorted_candidates, 1):
    print(f"  {rank}. {name:20s} {score:.4f}")

# Try blending top-2 candidates
top1_name = sorted_candidates[0][0]
top2_name = sorted_candidates[1][0]

best_alpha, best_blend_score = 1.0, all_scores[top1_name]
for alpha in np.arange(0.5, 1.01, 0.01):
    blended = alpha * candidates[top1_name][0] + (1 - alpha) * candidates[top2_name][0]
    s = competition_score(y_raw, blended)
    if s > best_blend_score:
        best_blend_score = s
        best_alpha = alpha

print(f"\nBest top-2 blend: alpha={best_alpha:.2f}, score={best_blend_score:.4f}")

if best_blend_score > all_scores[top1_name]:
    final_test_pred = np.expm1(
        best_alpha * candidates[top1_name][1] +
        (1 - best_alpha) * candidates[top2_name][1]
    )
    final_test_pred = np.clip(final_test_pred, 0, None)
    print("Using blended top-2 predictions!")
else:
    print("Single best candidate is better, keeping it.")

print(f"\nFinal prediction stats:")
print(f"  min:  {final_test_pred.min():.6f}")
print(f"  max:  {final_test_pred.max():.6f}")
print(f"  mean: {final_test_pred.mean():.6f}")

In [ ]:
best_lgb_model = lgb.LGBMRegressor(**best_lgb)
best_lgb_model.fit(X, y)

fi = (pd.DataFrame({"feature": X.columns,
                    "importance": best_lgb_model.feature_importances_})
        .sort_values("importance", ascending=False)
        .head(30))

plt.figure(figsize=(10, 9))
sns.barplot(data=fi, y="feature", x="importance", palette="viridis")
plt.title("Top-30 Feature Importances (LightGBM tuned)")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()

In [ ]:
submission = pd.DataFrame({
    "Index" : test["Index"].values,
    "demand": final_test_pred
})

assert submission.shape == (len(test), 2), "Shape mismatch!"
assert not submission["demand"].isna().any(), "NaN in predictions!"
assert (submission["demand"] >= 0).all(), "Negative predictions!"

submission.to_csv("submission.csv", index=False)
print("\nsubmission.csv saved!")
print(submission.head(10))
print(f"\nPrediction stats:\n{submission['demand'].describe()}")
print(f"\nSubmission shape: {submission.shape}")

In [ ]:
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"Features used: {len(FEATURE_COLS)}")
print(f"Models tuned: {list(tuned_models.keys())}")
print(f"Seeds used: {SEEDS}")
print(f"CV folds: {N_SPLITS}")
print(f"Best candidate: {best_name} (OOF={best_score:.4f})")
print(f"Submission shape: {submission.shape}")
print("="*60)